# EXPLORATORIO
## Wavelet Denoising

# 0. Importaciones y Configuración

In [1]:
from astropy.io import fits
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from astropy import constants as const
import os

import sys

sys.path.append('../../src')

from wavelet_denoising import Wavelet2D1DTransform, Denoiser2D1D

DATA_DIR = '/Users/kuky/Documents/practica/Denoiser3D-IFU/data'

mkdir -p failed for path /.matplotlib: [Errno 30] Read-only file system: '/.matplotlib'
Matplotlib created a temporary cache directory at /var/folders/2b/5tqr_dzn551f21glw27fr5t40000gn/T/matplotlib-i3ezj5ri because there was an issue with the default path (/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


# 1. Carga del cubo

In [6]:
CUBE_TYPE_DIR = 'real_cubes'
#CUBE_DIR = 'condor01comb'
CUBE_DIR = 'condor06ld'
#CUBE_NAME = 'CONDOR01COMB_CO32_15kms_r05._subcube1'
CUBE_NAME = 'CONDOR06LD_spw27_18kms_r05_subcube1'


In [7]:
hdu = fits.open(os.path.join(DATA_DIR, CUBE_TYPE_DIR, CUBE_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0

y = cube.copy() # Cubo de referencia de estadísticas. (No tenemos por lo que usamos el cubo original)

print(cube.shape)

(48, 96, 96)


In [8]:
hdu[0].header['BSCALE']

1.0

In [9]:
ref_freq = hdu[0].header['CRVAL3']
channel_width = hdu[0].header['CDELT3']
c = const.c.to('km/s').value

channel_width_km_s = channel_width * c / ref_freq

channel_width_km_s

np.float64(-17.410124561928097)

# 2. Denoiser

## 2.1. Soft Thresholding

In [6]:
denoiser_soft = Denoiser2D1D(threshold_type='soft', verbose=True)

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=cube.copy(),
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]

    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft.fits'), overwrite=True,output_verify='fix')

Number of 2D wavelet scales set to 6 (maximum value allowed by input image)
Number of 1D wavelet scales set to 5 (maximum value allowed by input image)

--- [ PERFORMING ITERATIVE DENOISING ] ---

----[ Denosiing with ITERATIVE SOFT THRESHOLDING ]----

[*] Trying with plateau condition: 1 consecutive stable residuals needed for convergence


--- [ DE-NOISING ITERATION #1 ] ---

(*) Decomposing noisy data into wavelet coefficients
(*) The gradient in the first iteration is 0
(*) Reconstructing the new signal coefficients into the real space
(*) Applying the positivity constraint
(*) Repeating these steps for subsequent iterations
(*) Aperture Flux: 1.693e+01, Clean Flux: 1.365e+01, Residual STD: 6.678e-05


--- [ DE-NOISING ITERATION #2 ] ---

(*) Updating model with gradient with respect to data
(*) Calculating weights for each iteration (except #1) to account for the soft thresholding bias
(*) Aperture Flux: 1.989e+01, Clean Flux: 1.365e+01, Residual STD: 5.871e-05
(*) Repeating these

## 2.2. Negative cubes and wavelets

In [7]:
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = cube * (-1)
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0

y = cube.copy() # Cubo de referencia de estadísticas. (No tenemos por lo que usamos el cubo original)

print(cube.shape)

(48, 96, 96)


In [8]:
denoiser_soft = Denoiser2D1D(threshold_type='soft', verbose=True)

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=cube.copy(),
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]

    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft_negative.fits'), overwrite=True,output_verify='fix')

Number of 2D wavelet scales set to 6 (maximum value allowed by input image)
Number of 1D wavelet scales set to 5 (maximum value allowed by input image)

--- [ PERFORMING ITERATIVE DENOISING ] ---

----[ Denosiing with ITERATIVE SOFT THRESHOLDING ]----

[*] Trying with plateau condition: 1 consecutive stable residuals needed for convergence


--- [ DE-NOISING ITERATION #1 ] ---

(*) Decomposing noisy data into wavelet coefficients
(*) The gradient in the first iteration is 0
(*) Reconstructing the new signal coefficients into the real space
(*) Applying the positivity constraint
(*) Repeating these steps for subsequent iterations
(*) Aperture Flux: 3.362e+00, Clean Flux: -1.365e+01, Residual STD: 1.441e-04


--- [ DE-NOISING ITERATION #2 ] ---

(*) Updating model with gradient with respect to data
(*) Calculating weights for each iteration (except #1) to account for the soft thresholding bias
(*) Aperture Flux: 4.927e+00, Clean Flux: -1.365e+01, Residual STD: 1.419e-04
(*) Repeating the

# 3. Testing modificaciones en los filtros y transformaciones

In [2]:
CUBE_NAME = 'CONDOR01COMB_CO32_15kms_r05._subcube1'
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0

y = cube.copy() # Cubo de referencia de estadísticas. (No tenemos por lo que usamos el cubo original)

print(cube.shape)

(48, 96, 96)


In [3]:
filtros = {
    'F_MALLAT_7_9': 1, # Biortogonal 7/9 [POR DEFECTO]
    'F_DAUBE_4': 2, # Daubechies 4
    'F_BI2HAAR': 3, # Biortogonal 2 Haar
    'F_BI4HAAR': 4, # Biortogonal 4 Haar
    'F_ODEGARD_7_9': 5, # Odégard 7/9
    'F_5_3': 6, # 5/3
    'F_LEMARIE_1': 7, # Lemarie orden 1
    'F_LEMARIE_3': 8, # Lemarie orden 3
    'F_LEMARIE_5': 9, # Lemarie orden 5
    'F_USER': 10, # Custom (requiere archivo de filtro)
    'F_HAAR': 11, # Haar
    'F_3_5': 12, # 3/5
    'F_4_4': 13, # 4/4
    # F_5_3_DIV (14) y F_MALLAT_9_7 (15) existen en el enum pero el binding
    # los rechaza (validacion: value < NBR_SB_FILTER=14). Valores validos: 1-13
}

transformaciones_1d = {
    'T01_PAVE_LINEAR': 0, # A trous lineal no decimada
    'T01_PAVE_B1SPLINE': 1, # A trous B1-spline no decimada
    'T01_PAVE_B3SPLINE': 2, # A trous B3-spline no decimada, Starlet 1D
    'T01_PAVE_B3_DERIV': 3, # A trous B3-spline derivada
    'T01_PAVE_HAAR': 4, # Haar no decimada
    'TM1_PAVE_MEDIAN': 5, # Mediana multiresolución
    'TU1_MALLAT':6, # Mallat no decimada
    'TU1_UNDECIMATED_NON_ORTHO':7, # No ortogonal no decimada
    'T01_PAVE_B3SPLINE_GEN2':8, # B3-spline generación 2
    'T01_PYR_B3SPLINE':9, # Pirámide B3-spline decimada
    'TM1_PYR_MEDIAN':10, # Pirámide mediana
    'T01_PAVE_MORLET':11, # Wavelet de Morlet no decimada
    'T01_PAVE_MEX':12, # Mexican Hat no decimada
    'T01_PAVE_FRENCH':13, # French Hat no decimada
    'T01_PAVE_DERIV_GAUSS':14, # Derivada de Gaussiana no decimada
    'T01_MALLAT':15, # Mallat decimada [POR DEFECTO]
    'T01_LIFTING':16, # Lifting scheme
    'WP1_MALLAT':17, # Wavelet Packets Mallat
    'WP1_LIFTING':18, # Wavelet Packets Lifting
    'WP1_ATROUS':19, # Wavelet Packets A trous
    'T01_PYR_LINEAR':20 # Pirámide lineal
}

transformaciones_2d = {
    'T01_PAVE_LINEAR':1, # A trous con kernel lineal
    'TO_PAVE_BSPLINE':2, # A trous con B-spline (Starlet) [POR DEFECTO]
    'TO_PAVE_FFT':3, # A trous usando FFT
    'TM_PAVE_MEDIAN':4, # A trous con mediana
    'TM_PAVE_MINMAX':5, # A trous Min-Max
    'TO_PYR_LINEAR':6, # Pirámide con kernel lineal
    'TO_PYR_BSPLINE':7, # Pirámide con B-spline
    'TO_PYR_FFT_DIFF_RESOL':8, #Pirámide FFT con diferencia de resolución
    'TO_PYR_MEYER':9, # Pirámide con wavelet de Meyer
    'TM_PYR_MEDIAN':10, # Pirámide con mediana
    'TM_PYR_LAPLACIAN':11, # Pirámide Laplaciana
    'TM_PYR_MINMAX':12, # Pirámide Min-Max
    'TM_PYR_SCALING_FUNCTION':13, # Pirámide con función de escala
    'TO_MALLAT':14, # Transformada de Mallat (ortogonal no decimada)
    'TO_FEAUVEAU':15, # Transformada de Feauveau
    'TO_PAVE_FEAUVEAU':16, # Transformada de Feauveau no decimada
    'TO_LC':17, # Line-Column
    'TO_HAAR':18, # Haar
    'TO_SEMI_PYR':19, # Semi-pirámide
    'TM_TO_SEMI_PYR':20, # Semi-pirámide (morpho)
    'TO_DIADIC_MALLAT':21, # Mallat diádica
    'TM_TO_PYR':22, # Pirámide morfológica
    'TO_PAVE_HAAR':23, # Haar no decimada
    'TO_UNDECIMATED_MALLAT':24, # Mallat no decimada
    'TO_UNDECIMATED_NON_ORTHO':25, # No decimada No ortogonal
    'TO_PYR_MEYER_ISOTROP':26, # Mater isotrópica en pirámide
    'TO_PYR_FFT_DIFF_SQUARE':27, # Pirámide FFT con diferencia cuadrada
    'TC_FCT':28, # Fast Curvelet Transform
    'TO_LIFTING':29 # Lifting scheme
}

In [4]:
T1D = 'T01_MALLAT'
T2D = 'TO_PAVE_BSPLINE'
F1D = 'F_MALLAT_7_9'
F2D = 'F_MALLAT_7_9'

denoiser = Denoiser2D1D(
    threshold_type='soft', # Tipo de umbral
    verbose=True, # Mostrar mensajes de progreso
    transform_1d=transformaciones_1d[T1D], # Transformación 1D
    transform_type=transformaciones_2d[T2D], # Tipo de transformación 2D
    filter_1d=filtros[F1D], # Filtro 1D
    filter_2d=filtros[F2D], # Filtro 2D
    )


result = denoiser.denoise(
    x=cube,
    y=cube.copy(),
    method='iterative',
    threshold_level=5
)
best_model = result[0]

hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
hdu[0].data = best_model
hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_{T1D}_{T2D}_{F1D}_{F2D}.fits'), overwrite=True,output_verify='fix')

Number of 2D wavelet scales set to 6 (maximum value allowed by input image)
Number of 1D wavelet scales set to 5 (maximum value allowed by input image)

--- [ PERFORMING ITERATIVE DENOISING ] ---

----[ Denoising with ITERATIVE SOFT THRESHOLDING ]----

[*] Trying with plateau condition: 1 consecutive stable residuals needed for convergence


--- [ DE-NOISING ITERATION #1 ] ---

(*) Decomposing noisy data into wavelet coefficients
(*) The gradient in the first iteration is 0
(*) Reconstructing the new signal coefficients into the real space
(*) Applying the positivity constraint
(*) Repeating these steps for subsequent iterations
(*) Aperture Flux: 7.083e+00, Clean Flux: 1.365e+01, Residual STD: 1.038e-04


--- [ DE-NOISING ITERATION #2 ] ---

(*) Updating model with gradient with respect to data
(*) Calculating weights for each iteration (except #1) to account for the soft thresholding bias
(*) Aperture Flux: 1.033e+01, Clean Flux: 1.365e+01, Residual STD: 8.739e-05
(*) Repeating these

# 4. Procedimiento en los 3 cubos de prueba.

## 4.1. PointingB

In [ ]:
CUBE_NAME = 'PointingB_calibrated_SourceA_Contsub_CubeLine_Natural_50kms_image_subcube1_subcube'

## -- Carga del cubo -- ##
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0
y = cube.copy() # Cubo de referencia de estadísticas. (No tenemos por lo que usamos el cubo original)


## -- Denoising -- ##
denoiser_soft = Denoiser2D1D(threshold_type='soft', verbose=True)

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=y,
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]
    
    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft.fits'), overwrite=True,output_verify='fix')


## -- Procedimiento con el cubo negativo -- ##
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = cube * (-1)
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0
y = cube.copy()

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=y,
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]
    
    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft_negative.fits'), overwrite=True,output_verify='fix')

Number of 2D wavelet scales set to 6 (maximum value allowed by input image)
Number of 1D wavelet scales set to 5 (maximum value allowed by input image)

--- [ PERFORMING ITERATIVE DENOISING ] ---

----[ Denosiing with ITERATIVE SOFT THRESHOLDING ]----

[*] Trying with plateau condition: 1 consecutive stable residuals needed for convergence


--- [ DE-NOISING ITERATION #1 ] ---

(*) Decomposing noisy data into wavelet coefficients
(*) The gradient in the first iteration is 0
(*) Reconstructing the new signal coefficients into the real space
(*) Applying the positivity constraint
(*) Repeating these steps for subsequent iterations
(*) Aperture Flux: 5.929e+00, Clean Flux: 3.384e+00, Residual STD: 4.754e-05


--- [ DE-NOISING ITERATION #2 ] ---

(*) Updating model with gradient with respect to data
(*) Calculating weights for each iteration (except #1) to account for the soft thresholding bias
(*) Aperture Flux: 7.804e+00, Clean Flux: 3.384e+00, Residual STD: 4.295e-05
(*) Repeating these

In [ ]:
CUBE_NAME = 'PointingB_calibrated_SourceA_Contsub_CubeLine_Natural_50kms_image_subcube1_subcube'
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = np.nan_to_num(hdu[0].data[0].astype(np.float64), nan=0.0) # Convertimos los datos a float64 y reemplazamos los NaN por 0.

ref_freq = hdu[0].header['CRVAL3']
channel_width = hdu[0].header['CDELT3']
c = const.c.to('km/s').value

channel_width_km_s = channel_width * c / ref_freq

print('channel_width_km_s', channel_width_km_s)

## -- Cálculo de SNR -- ##
rms_noise = np.std(cube[0:22])
peak_signal = np.max(cube)
peak_snr = peak_signal / rms_noise
print(f'Peak SNR: {peak_snr}')

channel_width_km_s -49.75107242809635
Peak SNR: 14.23531928619677


## 4.2. CONDOR06LD

In [ ]:
CUBE_NAME = 'CONDOR06LD_spw27_18kms_r05_subcube1'

## -- Carga del cubo -- ##
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0
y = cube.copy() # Cubo de referencia de estadísticas. (No tenemos por lo que usamos el cubo original)


## -- Denoising -- ##
denoiser_soft = Denoiser2D1D(threshold_type='soft', verbose=True)

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=y,
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]
    
    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft.fits'), overwrite=True,output_verify='fix')


## -- Procedimiento con el cubo negativo -- ##
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = cube * (-1)
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0
y = cube.copy()

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=y,
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]
    
    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft_negative.fits'), overwrite=True,output_verify='fix')

Number of 2D wavelet scales set to 6 (maximum value allowed by input image)
Number of 1D wavelet scales set to 5 (maximum value allowed by input image)

--- [ PERFORMING ITERATIVE DENOISING ] ---

----[ Denosiing with ITERATIVE SOFT THRESHOLDING ]----

[*] Trying with plateau condition: 1 consecutive stable residuals needed for convergence


--- [ DE-NOISING ITERATION #1 ] ---

(*) Decomposing noisy data into wavelet coefficients
(*) The gradient in the first iteration is 0
(*) Reconstructing the new signal coefficients into the real space
(*) Applying the positivity constraint
(*) Repeating these steps for subsequent iterations
(*) Aperture Flux: 9.670e+00, Clean Flux: 1.519e+00, Residual STD: 1.235e-04


--- [ DE-NOISING ITERATION #2 ] ---

(*) Updating model with gradient with respect to data
(*) Calculating weights for each iteration (except #1) to account for the soft thresholding bias
(*) Aperture Flux: 1.463e+01, Clean Flux: 1.519e+00, Residual STD: 1.132e-04
(*) Repeating these

## 4.3. CONDOR01COMB

In [ ]:
CUBE_NAME = 'CONDOR01COMB_CO32_15kms_r05._subcube1'

## -- Carga del cubo -- ##
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0
y = cube.copy() # Cubo de referencia de estadísticas. (No tenemos por lo que usamos el cubo original)


## -- Denoising -- ##
denoiser_soft = Denoiser2D1D(threshold_type='soft', verbose=True)

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=y,
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]
    
    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft.fits'), overwrite=True,output_verify='fix')


## -- Procedimiento con el cubo negativo -- ##
hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
cube = hdu[0].data[0]
cube = cube * (-1)
cube = np.nan_to_num(cube, nan=0.0) # Se reemplazan los NaN por 0
y = cube.copy()

for i in range(1,6):

    result = denoiser_soft.denoise(
        x=cube,
        y=y,
        method='iterative',
        threshold_level=i
    )
    best_model = result[0]
    
    hdu = fits.open(os.path.join(DATA_DIR, f'{CUBE_NAME}.fits'))
    hdu[0].data = best_model
    hdu.writeto(os.path.join(DATA_DIR, f'{CUBE_NAME}_wavelets_{i}_soft_negative.fits'), overwrite=True,output_verify='fix')

Number of 2D wavelet scales set to 6 (maximum value allowed by input image)
Number of 1D wavelet scales set to 5 (maximum value allowed by input image)

--- [ PERFORMING ITERATIVE DENOISING ] ---

----[ Denosiing with ITERATIVE SOFT THRESHOLDING ]----

[*] Trying with plateau condition: 1 consecutive stable residuals needed for convergence


--- [ DE-NOISING ITERATION #1 ] ---

(*) Decomposing noisy data into wavelet coefficients
(*) The gradient in the first iteration is 0
(*) Reconstructing the new signal coefficients into the real space
(*) Applying the positivity constraint
(*) Repeating these steps for subsequent iterations
(*) Aperture Flux: 1.693e+01, Clean Flux: 1.365e+01, Residual STD: 6.678e-05


--- [ DE-NOISING ITERATION #2 ] ---

(*) Updating model with gradient with respect to data
(*) Calculating weights for each iteration (except #1) to account for the soft thresholding bias
(*) Aperture Flux: 1.989e+01, Clean Flux: 1.365e+01, Residual STD: 5.871e-05
(*) Repeating these